<a href="https://colab.research.google.com/github/bastaps/algeria-tech-videos/blob/main/colab/ACE-Step-1.5_Colab_Qualite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 ACE-Step 1.5 QUALITÉ sur Colab — MusicGen Studio

Mode qualité : modèle **sft** (50 étapes) + **planification par le modèle de langage** (diction, placement des paroles). Plus lent que le notebook turbo (~3 à 6 min par morceau sur T4).

1. Menu **Exécution → Modifier le type d'exécution → T4 GPU** → **Enregistrer**.
2. Clique sur **▶** de la cellule ci-dessous. Une seule cellule, rien d'autre à faire.
3. Attends le **cadre vert** (15-20 min la 1re fois) et copie le lien `https://….gradio.live`.
4. Colle-le dans **MusicGen Studio → ⚙ Réglages → Lien ACE-Step**, moteur **Colab GPU**.

Laisse l'onglet ouvert pendant que tu utilises le studio.

In [ ]:
#@title ▶ Lancer ACE-Step 1.5 QUALITÉ (clique sur le bouton ▶ à gauche)
#@markdown Laisse les options par défaut. Tout est automatique : installation, modèles, puis lien vert.
import os, re, subprocess, sys, time, collections, threading, shutil
from IPython.display import HTML, display

COMMIT = "ca1e85fe9430179831e6bc6be790c332190a3866"   # version d'ACE-Step 1.5 testée : ne pas changer
REPO = "/content/ACE-Step-1.5"
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

def box(msg, ok=True):
    bg, bd = ("#0f3d2a", "#2ecc71") if ok else ("#4a1414", "#e74c3c")
    display(HTML(f'<div style="margin:10px 0;padding:14px 18px;border-radius:10px;background:{bg};'
                 f'border:2px solid {bd};color:#fff;font-family:sans-serif;font-size:15px">{msg}</div>'))

def stop(msg):
    box("❌ " + msg, ok=False)
    raise SystemExit(msg)

def sh(cmd, cwd=None, tries=1):
    for i in range(tries):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
        if r.returncode == 0:
            return r.stdout
        if i < tries - 1:
            print(f"   nouvel essai ({i + 2}/{tries})…"); time.sleep(10)
    print(r.stdout[-1500:]); print(r.stderr[-3000:])
    stop("Étape échouée : " + cmd.split()[0] + ". Envoie une capture de ce message.")

# 1) GPU
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    stop("Pas de GPU. Menu <b>Exécution → Modifier le type d'exécution → T4 GPU</b> → Enregistrer, puis relance ▶.")
print("✅ 1/4 GPU :", gpu)

# 2) Outils + code (version figée)
if not shutil.which("uv"):
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh", tries=3)
if not os.path.isdir(REPO + "/.git"):
    shutil.rmtree(REPO, ignore_errors=True)
    sh(f"git init -q {REPO} && cd {REPO} && git remote add origin https://github.com/ACE-Step/ACE-Step-1.5.git", tries=1)
    sh(f"git fetch -q --depth 1 origin {COMMIT} && git checkout -q FETCH_HEAD", cwd=REPO, tries=3)
print("✅ 2/4 Code ACE-Step 1.5 prêt")

# 3) Dépendances (environnement isolé : indépendant du Python de Colab)
if not os.path.exists(REPO + "/.venv/bin/acestep"):
    print("⏳ 3/4 Installation des dépendances (5 à 10 min la première fois)…")
    sh("uv sync --frozen", cwd=REPO, tries=3)
print("✅ 3/4 Dépendances installées")

# Correctif T4 : float16 donne des sons vides (NaN) sur les GPU anciens -> float32
orch = REPO + "/acestep/core/generation/handler/init_service_orchestrator.py"
src = open(orch, encoding="utf-8").read()
new = re.sub(r"(else:\n\s+self\.dtype = torch\.)float16(\n\s+logger\.info\(\n\s+\"\[initialize_service\] Pre-Ampere)",
             r"\1float32\2", src)
if new != src:
    open(orch, "w", encoding="utf-8").write(new)
elif "self.dtype = torch.float32\n                    logger.info(" not in src:
    stop("Correctif float32 introuvable.")

# 4) Serveur + lien
subprocess.run("pkill -f acestep_v15_pipeline; pkill -f '.venv/bin/acestep'", shell=True)
env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "MPLBACKEND")}
env.update(PYTHONUNBUFFERED="1", GRADIO_ANALYTICS_ENABLED="False", MPLBACKEND="Agg")
cmd = [REPO + "/.venv/bin/acestep", "--port", "7860", "--share", "--server-name", "127.0.0.1",
       "--init_service", "true", "--config_path", "acestep-v15-sft",
       "--init_llm", "true", "--lm_model_path", "acestep-5Hz-lm-0.6B", "--backend", "pt",
       "--offload_to_cpu", "true", "--quantization", "none"]
print("⏳ 4/4 Démarrage du serveur (le 1er lancement télécharge les modèles : 3 à 6 min)…")
proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)

tail = collections.deque(maxlen=60)
shown = False
KEEP = ("Error", "ERROR", "Traceback", "out of memory", "NaN")
try:
    for line in proc.stdout:
        tail.append(line.rstrip())
        m = re.search(r"https://[a-z0-9-]+\.gradio\.live", line)
        if m and not shown:
            shown = True
            box(f'✅ <b>ACE-Step 1.5 QUALITÉ est prêt.</b> Copie ce lien dans <b>MusicGen Studio → ⚙ Réglages → Lien ACE-Step</b> :'
                f'<div style="font-size:22px;font-weight:700;margin-top:8px;user-select:all;word-break:break-all">{m.group(0)}</div>'
                f'<div style="font-size:12px;margin-top:8px;opacity:.8">Le lien marche tant que cette cellule tourne '
                f'(icône qui tourne). Ne ferme pas cet onglet.</div>')
        elif any(k in line for k in KEEP):
            print(line, end="")
except KeyboardInterrupt:
    proc.terminate()
    print("⏹ Serveur arrêté.")

code = proc.wait()
if code not in (0, None, -15):
    print("\n".join(list(tail)[-30:]))
    box(f"⚠️ Le serveur s'est arrêté (code {code}). Envoie une capture de ce message, "
        "ou relance ▶ pour obtenir un nouveau lien.", ok=False)


✅ 1/4 GPU : Tesla T4, 15360 MiB
✅ 2/4 Code ACE-Step 1.5 prêt
⏳ 3/4 Installation des dépendances (5 à 10 min la première fois)…
